# Notebook 2
# Feature Engineering and Model Training

main objective to cover:
- Load the cleaned dataset
- Prepare features and target
- Split data into training and testing sets
- Build preprocessing pipeline
- Train multiple machine learning models
- Evaluate and compare the models
- Select the best model for deployment

## 2.1 Load Important libraries

In [27]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

## 2.2 Import Scikit-Learn Modules

In [28]:
# Train-test split
from sklearn.model_selection import train_test_split

# Feature scaling
from sklearn.preprocessing import StandardScaler

# Pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

## 2.3 Load the cleaned dataset

In [29]:
df = pd.read_csv("../data/processed/heart_cleaned.csv")
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


## Quick check Before building ML Model:

 - Do we have expected columns present?
 - Does the target column available?
 - Does the dataset have the expected number of rows and columns?

In [30]:
# Display the shape of the dataset
print("Dataset Shape:", df.shape)

# Display column names
print("\nColumn Names:")
print(df.columns.tolist())

Dataset Shape: (303, 14)

Column Names:
['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']


## 2.4 Separate Features (X) and Target (y)

In [31]:
# Separate features and target

X = df.drop("target", axis=1)
y = df["target"]

In [32]:
print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (303, 13)
Target Shape: (303,)


## 2.5 Split the Data into Training and Testing Sets

In [33]:
# Split the dataset into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2, #20% of the data goes to the testing set
    random_state=42, # To fix the split of the data each time(reproduceability)
    stratify=y # this will keep approximatly same class distribution in training and testing data
)
# X_train > Contains the features for training.
# X_test > Contains the features for testing.
# Y_train > Contains the Target data for training.
# Y_test > Contains the Target Data for testing.


In [34]:
print("Training Features Shape:", X_train.shape)
print("Testing Features Shape:", X_test.shape)

print("\nTraining Target Shape:", y_train.shape)
print("Testing Target Shape:", y_test.shape)

Training Features Shape: (242, 13)
Testing Features Shape: (61, 13)

Training Target Shape: (242,)
Testing Target Shape: (61,)


## 3.1 Feature Processing
 - We have features that have vast span and short span,they are not on the similar scale
 - Making features on a similar scale, allows algorithms like Logistic Regression to learn effectively
 - Random Forest doesn't require scaling because it's based on decision trees.
 - Before we proceed we will divide fearures in two groups, Neumerical feature and Categorical feature

In [35]:
# Define Fearure group
# Numerical features
numerical_features = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak"
]

# Categorical features
categorical_features = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal"
]

In [36]:
print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

Categorical Features:
['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']


## 3.2 Build the Preprocessing Pipeline
 - It is very important to keep same preprocessing, which must be applied during deployment. 
 - If the API scales data differently than the training process, predictions will be unreliable.

In [37]:
# Create a preprocessing pipeline

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features), # To apply standard scaler to numerical category
        ("cat", "passthrough", categorical_features) # It will leave the categorical ones unchanged
    ]
)
# By using ColumnTransformer inside a pipeline, 
# Scikit-learn guarantees that training and inference use the exact same preprocessing steps.

## 3.3 Create Logistic Regression Pipeline
 Combine, Preprocessor and Model to get Complete ML Pipeline

In [38]:
# Create Logistic Regression pipeline

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]
)

## 3.4 Create Random Forest Pipeline

In [39]:
# Create Random Forest pipeline

random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(random_state=42))
    ]
)

## 4.1 Train Machine Learning Models
 - Train Logistic Regression Model 
 - Train Random Forest Model

In [40]:
# Train Logistic Regression model
logistic_pipeline.fit(X_train, y_train)

# Train Random Forest model
random_forest_pipeline.fit(X_train, y_train)

print("Both models trained successfully.")

Both models trained successfully.


## 5.1 Make Predictions
 - Model Predicts the heart disease from the the patients data in our test set

In [41]:
# Logistic Regression predictions
logistic_predictions = logistic_pipeline.predict(X_test)

# Random Forest predictions
random_forest_predictions = random_forest_pipeline.predict(X_test)

In [42]:
# Probability predictions for ROC-AUC

logistic_probabilities = logistic_pipeline.predict_proba(X_test)[:, 1]

random_forest_probabilities = random_forest_pipeline.predict_proba(X_test)[:, 1]

## 5.2 Check for the predictions
 - Instead inspection all test samples we will look at first 10 predictioins, This is Slicing

In [43]:
print("Logistic Regression Predictions:")
print(logistic_predictions[:10])

print("\nRandom Forest Predictions:")
print(random_forest_predictions[:10])

Logistic Regression Predictions:
[0 1 0 0 1 0 0 0 1 0]

Random Forest Predictions:
[0 1 0 0 0 0 0 0 1 0]


 - The Prediction made by two model for Patient No 5 is different
 - In order to confirm whjhich model is correct, we evaluate the models

## 5.3 Evaluate the Models

In [44]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [45]:
def evaluate_model(model_name, y_true, y_pred, y_prob):
    """
    Evaluate a classification model and return key performance metrics.
    """

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_prob)

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    }

In [46]:
# Evaluate Logistic Regression

logistic_results = evaluate_model(
    "Logistic Regression",
    y_test,
    logistic_predictions,
    logistic_probabilities
)

# Evaluate Random Forest

random_forest_results = evaluate_model(
    "Random Forest",
    y_test,
    random_forest_predictions,
    random_forest_probabilities
)

# Comparison table

model_results = pd.DataFrame([
    logistic_results,
    random_forest_results
])

model_results

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.868852,0.812500,0.928571,0.866667,0.951299
1,Random Forest,0.918033,0.870968,0.964286,0.915254,0.954004


### Model Comparison Summary

Both Logistic Regression and Random Forest achieved strong performance on the test dataset. However, Random Forest outperformed Logistic Regression across all evaluation metrics, including Accuracy, Precision, Recall, F1 Score, and ROC-AUC.

Therefore, the Random Forest model was selected as the preferred model for further validation, experiment tracking, model packaging, and deployment.